In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# cmfe(기존 LTAU 기반)와 cue4unc(MC Dropout / Deep Ensemble)의 결과를 함께 비교한다.
# 세 결과 모두 동일한 CUE 아키텍처 + 동일한 5-fold 내부 CV 파이프라인에서 나온 것이며,
# 유일한 차이는 modality 불확실성을 계산하는 방법(u_m(x))이다:
#   - cmfe:            LTAU (loss-trajectory 기반 k-NN 불확실성)
#   - cue4unc/MC_Dropout:  MC Dropout (T회 확률적 forward pass의 예측 분산)
#   - cue4unc/Deep_Ensemble: Deep Ensemble (서로 다른 seed로 학습한 독립 모델들의 예측 분산)
cue_base = '/data/rlawlsgurjh/work/gitlab/CUE/cue/results'
cue4unc_base = '/data/rlawlsgurjh/work/gitlab/CUE/cue4unc/results'

datasets = ['DIS', 'INH']
# cue4unc의 run_cmfe_mc_dropout.sh / run_cmfe_ensemble.sh가 학습하는 타겟과 동일하게 맞춘다.
targets = ['PI', 'Ssel', 'WS2', 'RS2']
metrics_of_interest = ['rmse', 'r2', 'pcc']

# 비교할 세 가지 소스: (표시 이름, metrics.csv 루트 경로)
# 실제 결과 폴더명은 여전히 'CUE'이므로 경로는 그대로 두고, 표시 이름만 CUE로 바꾼다.
model_sources = {
    'CUE (LTAU)': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'CUE'),
    'CUE (MC Dropout)': os.path.join(cue4unc_base, 'MC_Dropout', '{dataset}', '{target}', 'v1', 'CUE'),
    'CUE (Deep Ensemble)': os.path.join(cue4unc_base, 'Deep_Ensemble', '{dataset}', '{target}', 'v1', 'CUE'),
}

# 그래프에 표시될 모델 순서
model_order = ['CUE (MC Dropout)', 'CUE (Deep Ensemble)', 'CUE (LTAU)']

In [2]:
def load_metric(model_dir_template, dataset, target):
    """cue/cue4unc의 metrics.csv(train.py::aggregate_metrics 출력)를 읽어
    model_name == 'mlp' 행의 rmse/r2/pcc(+ _std)를 뽑아온다.
    cmfe와 cue4unc 모두 동일한 aggregate_metrics 함수를 사용하므로 스키마가 동일하다."""
    metrics_path = os.path.join(model_dir_template.format(dataset=dataset, target=target), 'metrics.csv')
    if not os.path.exists(metrics_path):
        return None

    df_m = pd.read_csv(metrics_path)
    mlp_row = df_m[df_m['model_name'] == 'mlp']
    if mlp_row.empty:
        return None

    values = {}
    for col in mlp_row.columns:
        if col in ['model_name', 'Unnamed: 0']:
            continue
        norm_col = col.lower()
        if norm_col in metrics_of_interest or norm_col.replace('_std', '') in metrics_of_interest:
            values[norm_col] = mlp_row[col].values[0]
    return values if values else None


raw_rows = []
for d_name in datasets:
    for target in targets:
        for model_name, dir_template in model_sources.items():
            vals = load_metric(dir_template, d_name, target)
            if vals is not None:
                raw_rows.append({'Dataset': d_name, 'Target': target, 'Model': model_name, **vals})

df_total = pd.DataFrame(raw_rows)
print(f"✅ 데이터 로드 완료: {len(df_total)}개의 항목이 수집되었습니다.")

✅ 데이터 로드 완료: 24개의 항목이 수집되었습니다.


In [ ]:
METRIC_HEADERS = {'rmse': 'RMSE( ↓ )', 'r2': 'R2( ↑ )', 'pcc': 'PCC( ↑ )'}


def build_metric_table(df_all, dataset, target, metrics=metrics_of_interest, model_order=model_order):
    """지정한 dataset/target에 대해 Model별 metric 값('mean ± std')을 담은 표를 만든다 (순위 없음).

    Args:
        df_all (pd.DataFrame): load_metric()으로 수집한 전체 결과(df_total).
        dataset (str): 'DIS' | 'INH'.
        target (str): 'PI' | 'Ssel' | 'WS2' | 'RS2'.
        metrics (list[str]): 값을 담을 metric 목록 (순서대로 rmse, r2, pcc).
        model_order (list[str]): 비교할 모델(행) 순서. 기본값은 전역 model_order이며,
            다른 모델 집합(예: 버전 비교)을 표로 만들 때 별도로 넘길 수 있다.

    Returns:
        Optional[pd.DataFrame]: 컬럼이 Model | RMSE( ↓ ) | R2( ↑ ) | PCC( ↑ ) 순서인 표.
            각 값은 '{mean:.3f} ± {std:.3f}' 문자열이다.
            해당 dataset/target에 데이터가 하나도 없으면 None.
    """
    df_sub = df_all[(df_all['Dataset'] == dataset) & (df_all['Target'] == target)]
    df_sub = df_sub.set_index('Model').reindex(model_order)
    df_sub = df_sub.dropna(subset=[m for m in metrics if m in df_sub.columns], how='all')
    if df_sub.empty:
        return None

    table = pd.DataFrame(index=df_sub.index)
    for m in metrics:
        if m not in df_sub.columns:
            continue
        header = METRIC_HEADERS.get(m, m.upper())
        if f'{m}_std' in df_sub.columns:
            table[header] = [f"{v:.3f} ± {s:.3f}" for v, s in zip(df_sub[m], df_sub[f'{m}_std'])]
        else:
            table[header] = df_sub[m].round(3)

    return table.reset_index()  # 'Model'을 첫 컬럼으로 복원


table_save_dir = os.path.join(cue_base, 'table', 'uncertainty_comparison')
os.makedirs(table_save_dir, exist_ok=True)

# DIS -> dis(dissociation), INH -> inh(inhibition)
dataset_filenames = {'DIS': 'dis.csv', 'INH': 'inh.csv'}

metric_tables = {}
for d_name in datasets:
    dataset_tables = []
    for target in targets:
        table = build_metric_table(df_total, d_name, target)
        if table is None:
            continue
        metric_tables[(d_name, target)] = table

        print(f"\n=== [{d_name} / {target}] ===")
        print(table.to_string(index=False))

        # 맨 처음 컬럼에 Target을 추가한 뒤(targets 순서 = PI, Ssel, WS2, RS2) 이어 붙인다.
        table_with_target = table.copy()
        table_with_target.insert(0, 'Target', target)
        dataset_tables.append(table_with_target)

    if not dataset_tables:
        continue

    dataset_table = pd.concat(dataset_tables, ignore_index=True)
    save_path = os.path.join(table_save_dir, dataset_filenames[d_name])
    dataset_table.to_csv(save_path, index=False, encoding='utf-8-sig', sep='\t')
    print(f"\n✅ 저장 완료: {save_path}")

In [ ]:
# CUE (MC Dropout)의 서로 다른 실행 버전(v1/v2/v3, dropout_rate 등 하이퍼파라미터 차이)을 비교한다.
# 위 표와 달리 저장하지 않고 출력만 한다.
mc_version_sources = {
    'CUE (MC Dropout v1)': os.path.join(cue4unc_base, 'MC_Dropout', '{dataset}', '{target}', 'v1', 'CUE'),
    'CUE (MC Dropout v2)': os.path.join(cue4unc_base, 'MC_Dropout', '{dataset}', '{target}', 'v2', 'CUE'),
    'CUE (MC Dropout v3)': os.path.join(cue4unc_base, 'MC_Dropout', '{dataset}', '{target}', 'v3', 'CUE'),
}
mc_version_order = list(mc_version_sources.keys())

raw_rows_mc_versions = []
for d_name in datasets:
    for target in targets:
        for model_name, dir_template in mc_version_sources.items():
            vals = load_metric(dir_template, d_name, target)
            if vals is not None:
                raw_rows_mc_versions.append({'Dataset': d_name, 'Target': target, 'Model': model_name, **vals})

df_mc_versions = pd.DataFrame(raw_rows_mc_versions)
print(f"✅ 데이터 로드 완료: {len(df_mc_versions)}개의 항목이 수집되었습니다.")

for d_name in datasets:
    for target in targets:
        table = build_metric_table(df_mc_versions, d_name, target, model_order=mc_version_order)
        if table is None:
            continue

        print(f"\n=== [{d_name} / {target}] CUE (MC Dropout) v1 vs v2 vs v3 ===")
        print(table.to_string(index=False))